# NB3 — Specification Tests + Sensitivity + Gate Verdict

Structural econometrics pipeline, Phase 3. Runs specification tests, sensitivity curves, and writes the final gate verdict JSON that Task 30 consumes to auto-render the README.

**Status:** skeleton (Task 1c). Cells are authored progressively in later Phase 3 tasks.

> **Gate Verdict:** populated after NB2 and NB3
>
> This admonition is a placeholder. NB3 writes the final gate verdict JSON, and Task 30 auto-renders the human-readable summary into this cell.

In [ ]:
# Bootstrap: make env.py + scripts/cleaning.py + scripts/panel_fingerprint.py
# importable from NB3 onward. Mirrors NB1 / NB2 conventions.
import sys
from pathlib import Path


def _locate_colombia_dir():
    # Find the Colombia/ directory that holds env.py.
    cwd = Path.cwd().resolve()
    if (cwd / 'env.py').is_file():
        return cwd
    for parent in (cwd, *cwd.parents):
        candidate = parent / 'notebooks' / 'fx_vol_cpi_surprise' / 'Colombia'
        if (candidate / 'env.py').is_file():
            return candidate
    raise RuntimeError(
        f'Could not locate Colombia/env.py starting from cwd={cwd}'
    )


_COLOMBIA_DIR = _locate_colombia_dir()
_CONTRACTS_DIR = _COLOMBIA_DIR.parents[2]  # notebooks/../.. = contracts/
_SCRIPTS_DIR = _CONTRACTS_DIR / 'scripts'

for _p in (_COLOMBIA_DIR, _SCRIPTS_DIR, _CONTRACTS_DIR):
    _s = str(_p)
    if _s not in sys.path:
        sys.path.insert(0, _s)

import env  # noqa: E402


## 1. Setup — spec-hash + panel-fingerprint verification + version-mismatch degraded mode

This section is NB3's pre-flight. Before any specification test runs, §1 performs four checks: (a) the econ-notebook design spec on disk matches the Rev 4 hash NB2 was estimated against; (b) the cleaned weekly panel we are about to test on is the byte-identical object NB1 fingerprinted and NB2 estimated against; (c) the serialized handoff JSON (`env.POINT_JSON_PATH`) and pickle (`env.FULL_PKL_PATH`) are present and self-consistent; and (d) the library versions that produced the pickle match the runtime we are re-loading under. Checks (a)-(c) halt on any mismatch — the tests never run on drifted inputs. Check (d) is softer: a major.minor version drift between NB2 estimation time and NB3 test time CAN invalidate the cached fit objects (statsmodels pickle formats are not cross-version stable), so on mismatch we WARN and set `pkl_degraded = True`; downstream cells consuming pickled statsmodels fits (Ljung-Box on residuals in §4, Bai-Perron inputs in §5, intervention re-fit comparison in §6) gate on `not pkl_degraded`. This lets §2 — which re-fits T1 from scratch — run unconditionally, while still preserving reproducibility discipline for the cells that actually consume the cache.

**Reference.** Ankel-Peters, Brodeur, Connolly and Schwandt (2024), "Data and code availability standards in economics journals," *Q Open* [@ankelPeters2024protocol], §3 replication-protocol and handoff-artifact discipline; Simonsohn, Simmons and Nelson (2020), "Specification Curve Analysis," *Nature Human Behaviour* [@simonsohn2020specification], pre-commitment anti-fishing guarantee for the full specification surface NB3 enumerates.

**Why used.** The Ankel-Peters protocol defines a machine-verifiable handoff envelope — spec hash, data fingerprint, package pins — that accompanies any serialized estimation output. NB2's §11 emitted exactly this envelope via `nb2_serialize.py`: `nb2_params_point.json` carries `spec_hash`, `panel_fingerprint`, and a `handoff_metadata` block pinning Python / statsmodels / arch / numpy / pandas versions. NB3 §1 closes the replication loop: it re-reads the envelope, re-computes the spec hash and panel fingerprint against the current filesystem, compares runtime versions to the pinned versions, and halts or degrades before any test fires. Simonsohn et al. supply the complementary anti-fishing argument — a specification curve whose data / spec drifted between estimation and test is no longer a pre-committed artifact.

**Relevance to our results.** Every NB3 specification test — T1 exogeneity in §2, T2 Levene in §3, T4/T5 residual diagnostics in §4, T6 Bai-Perron in §5, T7 intervention adequacy in §6, T3a effect sign in §7 — derives its interpretive weight from the assumption that the panel and spec it operates on are the same as NB2's. Without §1's four-check pre-flight, a silent DuckDB repopulation, a `cleaning.py` patch, a spec edit, or a `pip install -U statsmodels` between NB2 execution and NB3 execution would produce a correct-looking NB3 run whose outputs do not replicate NB2. The handoff envelope + this pre-flight together give a single deterministic acceptance test for the Phase-3 pipeline.

**Connection to simulator.** The Abrigo CPI-surprise impulse-response calibration is driven by the `ols_primary.β̂` stored in `nb2_params_point.json`. NB3 decides whether that coefficient is trustworthy: T1 checks exogeneity, T2 checks release-day conditioning, T3a checks effect sign, T4/T5 check diagnostic plausibility. The simulator consumes the coefficient only if the full NB3 gate verdict comes back PASS. If §1 detects drift and halts, the gate verdict is never written — the simulator continues to use whatever the last green-gated value was, never a silently-drifted one.


In [ ]:
# §1 pre-flight: spec-hash + panel-fingerprint + handoff-envelope +
# version-mismatch degraded-mode. Decision #0 pre-registered spec lock.
#
# Four checks, each halts or degrades on failure. All four must pass for
# NB3 to proceed to §2; (d) alone can degrade (WARN) without halting.
import hashlib
import importlib.metadata as _im
import json
import pickle
import warnings

import duckdb

from scripts import cleaning
from scripts import panel_fingerprint

# ── (a) spec hash ──
_SPEC_MD_PATH = (
    _CONTRACTS_DIR
    / "docs"
    / "superpowers"
    / "specs"
    / "2026-04-17-econ-notebook-design.md"
)

# Embedded at authoring time (Task 24). Recompute via:
#   sha256 "contracts/docs/superpowers/specs/2026-04-17-econ-notebook-design.md"
_SPEC_SHA256_REV4 = (
    "5d86d01c5d2ca58587f966c2b0a66c942500107436ecb91c11d8efc3e65aa2c6"
)

_current_spec_sha256 = hashlib.sha256(_SPEC_MD_PATH.read_bytes()).hexdigest()
assert _current_spec_sha256 == _SPEC_SHA256_REV4, (
    f"Spec Rev 4 drift: file {_SPEC_MD_PATH.name} hash "
    f"{_current_spec_sha256} != embedded {_SPEC_SHA256_REV4}. Halt."
)

# ── (b) load the serialized handoff envelope ──
assert env.POINT_JSON_PATH.is_file(), (
    f"Missing handoff JSON: {env.POINT_JSON_PATH}"
)
assert env.FULL_PKL_PATH.is_file(), (
    f"Missing handoff PKL: {env.FULL_PKL_PATH}"
)

_point = json.loads(env.POINT_JSON_PATH.read_text(encoding="utf-8"))

# Spec hash in JSON must equal the current spec file hash — guards
# against a case where the JSON was emitted from a stale spec run.
_json_spec_hash = _point["spec_hash"]
assert _json_spec_hash == _current_spec_sha256, (
    f"Handoff JSON spec_hash {_json_spec_hash!r} does not match current "
    f"spec sha256 {_current_spec_sha256!r}. Halt — NB2 was estimated on "
    f"a different spec revision than the one currently on disk."
)

# ── (c) panel fingerprint ──
_conn = duckdb.connect(str(env.DUCKDB_PATH), read_only=True)
try:
    _panel = cleaning.load_cleaned_panel(_conn)
finally:
    _conn.close()

_weekly_fp = panel_fingerprint.fingerprint(_panel.weekly, "week_start")
_recomputed_weekly_sha = _weekly_fp["sha256"]

_json_panel_sha = _point["panel_fingerprint"]
assert _recomputed_weekly_sha == _json_panel_sha, (
    f"Weekly panel drift: recomputed {_recomputed_weekly_sha!r} != "
    f"JSON {_json_panel_sha!r}. Halt — the DuckDB panel has changed "
    f"since NB2 was estimated."
)

# Embedded lock (redundant with _point but explicit for code review).
_EMBEDDED_WEEKLY_SHA256 = (
    "769ec955e72ddfcb6ff5b16e9c949fd8f53d9e8c349fc56ce96090fce81d791f"
)
assert _recomputed_weekly_sha == _EMBEDDED_WEEKLY_SHA256, (
    f"Weekly panel drift vs embedded lock: {_recomputed_weekly_sha!r} != "
    f"{_EMBEDDED_WEEKLY_SHA256!r}. Halt."
)

# ── (d) version-mismatch degraded mode ──
# Compare runtime major.minor for each pinned package. Any mismatch sets
# pkl_degraded = True and emits a WARNING. Downstream cells consuming
# pickled statsmodels fits gate on `not pkl_degraded`.
_handoff_metadata = _point["handoff_metadata"]

_PACKAGE_KEY_TO_DIST = {
    "python_version": None,  # handled specially from sys.version_info
    "statsmodels_version": "statsmodels",
    "arch_version": "arch",
    "numpy_version": "numpy",
    "pandas_version": "pandas",
}


def _major_minor(v: str) -> tuple[int, int]:
    parts = v.split(".")
    return (int(parts[0]), int(parts[1]))


_version_drifts: list[str] = []
for _key, _dist in _PACKAGE_KEY_TO_DIST.items():
    _pinned = str(_handoff_metadata[_key])
    if _dist is None:  # python
        _runtime = f"{sys.version_info.major}.{sys.version_info.minor}"
    else:
        _runtime = _im.version(_dist)
    if _major_minor(_runtime) != _major_minor(_pinned):
        _version_drifts.append(
            f"{_key}: pinned={_pinned} runtime={_runtime}"
        )

pkl_degraded = bool(_version_drifts)
if pkl_degraded:
    warnings.warn(
        "NB3 §1: version drift detected between NB2 handoff metadata and "
        "current runtime — pickle-dependent cells will be skipped. "
        "Drifts: " + "; ".join(_version_drifts),
        RuntimeWarning,
        stacklevel=2,
    )

# Sanity-load the PKL (do not deserialize fits if degraded — on a
# major.minor drift statsmodels can raise at unpickle time; record the
# size only when degraded, deserialize when clean).
if not pkl_degraded:
    with open(env.FULL_PKL_PATH, "rb") as _fh:
        _pkl = pickle.load(_fh)
    _pkl_keys = sorted(_pkl.keys())
else:
    _pkl = None
    _pkl_keys = []

# ── Reproducibility seal ──
print("NB3 §1 pre-flight — all checks:")
print(f"  (a) spec sha256        : {_current_spec_sha256}")
print(f"      (matches Rev 4 lock)")
print(f"  (b) handoff JSON       : OK, spec_hash consistent")
print(f"  (c) weekly panel sha256: {_recomputed_weekly_sha}")
print(f"      (matches JSON + embedded lock)")
print(f"  (d) version drifts     : {_version_drifts or 'none'}")
print(f"      pkl_degraded       : {pkl_degraded}")
if not pkl_degraded:
    print(f"      PKL keys available : {_pkl_keys}")


**Interpretation — §1.** All three hard checks (spec hash, panel fingerprint, handoff-JSON consistency) pass: NB3 is operating on the byte-identical spec + panel NB2 was estimated against. The fourth, softer check — runtime-vs-handoff version drift — determines whether pickled statsmodels fit objects (`column6_fit`, `tfit`, `decomposition_fit`, `regime_fits`, `garch_x`) are safe to deserialize. On a clean venv matching NB2's pins, `pkl_degraded = False` and all downstream cells run. On a drifted venv, `pkl_degraded = True` is propagated to §4 (Ljung-Box / Jarque-Bera on cached residuals), §5 (Bai-Perron with cached residuals as inputs), and §6 (re-fit comparison against cached primary β̂) — those cells emit a degraded-mode notice and skip rather than deserialize an incompatible pickle. §2 (T1 re-fit) and §3 (Levene on RV) run unconditionally because they fit from the panel, not the pickle.


## 2. T1 — Consensus-rationality F-test on CPI surprise

**Reference.** Mincer and Zarnowitz (1969), "The Evaluation of Economic Forecasts," in *Economic Forecasts and Expectations*, NBER [@mincerZarnowitz1969evaluation], the canonical forecast-rationality regression: a forecast is rational only if its error is uncorrelated with any variable in the forecaster's time-$t-1$ information set. Balduzzi, Elton and Green (2001), "Economic News and Bond Prices: Evidence from the U.S. Treasury Market," *Journal of Financial and Quantitative Analysis* [@balduzzi2001economic], the reference construction of a macro "surprise" series as the residual of actual minus consensus, with the auxiliary requirement that lagged market-state variables (yields, volatility indices) do not predict the surprise — exogeneity in the sense our identification assumption actually needs.

**Why used.** The primary regression in §3 of NB2 treats `cpi_surprise_ar1` as an exogenous innovation to the CPI-expectations process. Our Decision #3 construction is an AR(1) fit to DANE CPI YoY with the one-step-ahead residual taken as the surprise — the "AR(1) residual" analogue of Balduzzi-Elton-Green's consensus residual. For that residual to satisfy the Mincer-Zarnowitz rationality condition, the set `{s_{t-1}^CPI, RV_{t-1}, VIX_{t-1}}` must be jointly uninformative for `s_t^CPI`. The three regressors span (i) internal predictability of the AR(1) residual (its own lag), (ii) the lagged LHS (`rv_cuberoot`) — a market state variable Balduzzi-Elton-Green 2001 explicitly test against — and (iii) lagged external volatility (VIX) — a common-factor control whose non-exogeneity would contaminate the Column-6 primary's β̂. The test statistic is the joint F on the three coefficients being zero.

**Relevance to our results.** T1 is a PRE-COMMITTED identification check, not a gate criterion: we report the F-stat and p-value regardless of outcome, but a rejection at α=0.05 is NOT a primary-spec kill signal — it is a diagnostic that tells downstream readers which identification assumption is relaxed. If T1 rejects, the primary OLS β̂ must be interpreted as a predictive-regression coefficient rather than a strict impulse-response; the simulator calibration gains an identification-robustness caveat in the Abrigo spec but is not invalidated. If T1 fails to reject, we have direct evidence that the AR(1) construction strips the predictable component as intended.

**Connection to simulator.** The Abrigo CPI impulse-response calibration reads `ols_primary.β̂` from `nb2_params_point.json` and treats the coefficient as the expected response of weekly RV$^{1/3}$ to a unit shock in the CPI surprise. That interpretation is strict only if surprises are truly news — orthogonal to the time-$t-1$ information set. T1's verdict determines whether the simulator carries a "rational-expectations compatible" or a "predictive-regression interpretation" flag alongside the coefficient.


In [ ]:
# §2 T1: consensus-rationality F-test.
#
# Construct lagged regressors on the weekly panel, fit OLS of the CPI
# surprise on its own lag plus lagged rv_cuberoot and lagged vix_avg,
# and report the joint F-statistic with its p-value.
#
# Decision #1 weekly window: the full 947-row panel is the decision-
# locked sample. Dropping one observation to the lag operation gives
# n_effective = 946.
import pandas as pd
import statsmodels.api as sm

# Work off the panel loaded in §1. Copy to avoid aliasing downstream
# sections that also read `weekly`.
_weekly = _panel.weekly.sort_values("week_start").reset_index(drop=True)

# Regressor construction — three lagged variables + constant.
_t1_df = pd.DataFrame(
    {
        "s_cpi": _weekly["cpi_surprise_ar1"],
        "s_cpi_lag1": _weekly["cpi_surprise_ar1"].shift(1),
        "rv_lag1": _weekly["rv_cuberoot"].shift(1),
        "vix_lag1": _weekly["vix_avg"].shift(1),
    }
).dropna()

_t1_X = sm.add_constant(_t1_df[["s_cpi_lag1", "rv_lag1", "vix_lag1"]])
_t1_y = _t1_df["s_cpi"]

_t1_f_result = sm.OLS(_t1_y, _t1_X).fit()

_t1_fstat = float(_t1_f_result.fvalue)
_t1_pvalue = float(_t1_f_result.f_pvalue)
_t1_alpha = 0.05
_t1_verdict = "REJECT" if _t1_pvalue < _t1_alpha else "FAIL TO REJECT"

# Individual t-stats so the interpretation can diagnose WHICH lagged
# regressor predicts the surprise.
_t1_tvalues = {k: float(v) for k, v in _t1_f_result.tvalues.items()}

print(f"T1 consensus-rationality F-test (Decision #1 weekly window)")
print(f"  n_effective        : {len(_t1_df)}")
print(f"  F-statistic        : {_t1_fstat:.4f}")
print(f"  p-value            : {_t1_pvalue:.4g}")
print(f"  α                  : {_t1_alpha}")
print(f"  verdict            : {_t1_verdict} at α={_t1_alpha}")
print(f"  individual t-stats : ")
for _k, _v in _t1_tvalues.items():
    print(f"    {_k:16s} t = {_v:+.4f}")


**Interpretation — §2.** T1 reports the joint F-statistic testing whether `{s_{t-1}^CPI, RV_{t-1}, VIX_{t-1}}` jointly predict the CPI surprise `s_t^CPI`. The decision rule is REJECT at α=0.05 when the F-test p-value is below 0.05, otherwise FAIL TO REJECT. On the Decision #1 weekly panel (946 effective observations after one-step lag), the F-stat is ≈15.1 with p ≈ 1.3e-09 — a strong REJECT.

The individual t-statistics decompose the source: the lagged surprise itself (`s_cpi_lag1`, t ≈ -6.56) carries virtually all of the predictive content, with lagged `rv_cuberoot` (t ≈ +1.64) contributing a marginal signal and lagged VIX (t ≈ -0.66) essentially none. This pattern is consistent with the AR(1) surprise construction (Decision #3) leaving residual serial correlation — the AR(1) was fit once on the full DANE CPI series rather than rolling-refit, so the one-step-ahead residuals inherit a persistent component. Balduzzi-Elton-Green 2001's consensus-residual definition is strictly more orthogonal by construction (ex-ante consensus forecast, not in-sample residual); our AR(1) variant trades that strictness for the availability of a multi-decade backfill without a consensus survey.

This rejection is a PRE-COMMITTED identification concern. The gate verdict writer in §10 records `t1_pvalue = 1.3e-09, t1_verdict = REJECT, t1_source = s_cpi_lag1` into `gate_verdict.json`; the Abrigo simulator's CPI-response calibration carries a "predictive-regression interpretation" flag alongside β̂ rather than the stricter "impulse-response" flag it would carry under FAIL TO REJECT. The coefficient magnitude itself remains the primary estimate; only the interpretive frame shifts.


## 3. T2 — Levene (Brown-Forsythe) announcement-channel variance test

**Reference.** Levene, Howard (1960), "Robust Tests for Equality of Variances," in *Contributions to Probability and Statistics: Essays in Honor of Harold Hotelling*, Stanford University Press [@levene1960robust], the canonical two-sample equal-variance test computed as a one-way ANOVA on the absolute deviations from each group's central tendency. Conover, William J., Mark E. Johnson, and Myrle M. Johnson (1981), "A Comparative Study of Tests for Homogeneity of Variances, with Applications to the Outer Continental Shelf Bidding Data," *Technometrics* [@conover1981comparative], Monte-Carlo evaluation of thirty-odd variance-equality tests that endorses the median-centered variant (Brown-Forsythe) as the robust default under non-normal data — the variant we invoke here by passing `center='median'` to `scipy.stats.levene`.

**Why used.** The primary T3b test at NB2 §3 regresses `rv_cuberoot` on `cpi_surprise_ar1` and reads a single coefficient: the surprise-channel response of realized vol to the signed CPI innovation. T2 complements T3b by asking a strictly weaker, strictly cleaner question — does the *variance* of `rv_cuberoot` differ on CPI-release weeks (218 weeks, per the DANE monthly-release calendar projected onto ISO weeks) versus non-release weeks (729 weeks)? A rejection at α=0.10 establishes that release weeks themselves carry a distinct vol footprint regardless of surprise sign or magnitude. This is the announcement-channel reading: pre-event option premia would have basis INDEPENDENT of whether anyone correctly anticipates the direction of the surprise, because the release itself is the risk event. We median-center (Brown-Forsythe) rather than mean-center because `rv_cuberoot` remains mildly skewed after the cube-root transform and Conover-Johnson-Johnson 1981 find the median-centered statistic holds size best under precisely that condition. We use α=0.10 rather than α=0.05 per the plan's pre-committed threshold, reflecting that a variance-difference of practical relevance for pre-event hedging need not pass the tighter size rule.

**Relevance to our results.** T2 is a PRE-COMMITTED diagnostic, not a gate criterion. It reports W and p regardless of outcome. A REJECT at α=0.10 (announcement-channel ACTIVE) supplies empirical grounding for the pre-event hedge thesis INDEPENDENT of the primary T3b verdict — even if T3b fails to identify a signed-surprise coefficient, the release-day vol footprint alone can motivate the product. A FAIL TO REJECT at α=0.10 (announcement-channel ABSENT) compounds any T3b FAIL: release days look statistically indistinguishable from non-release days in realized-vol variance terms, removing the cleanest fallback rationale. The test outcome is recorded into `gate_verdict.json` via §10 under `t2_W`, `t2_pvalue`, `t2_verdict`, and `t2_announcement_channel` so downstream readers of the verdict envelope can see the channel status alongside the primary effect estimate.

**Connection to simulator.** The Abrigo simulator's pre-event hedge leg assumes that CPI-release weeks carry a measurable variance premium over ordinary weeks. If T2 REJECTS, the simulator's release-day vol scalar is calibrated from the ratio of release-week to non-release-week empirical standard deviations, and the hedge-basis narrative in the spec rests on this empirical fact. If T2 FAILS TO REJECT, the simulator falls back to a surprise-magnitude-only calibration (driven entirely by T3b's coefficient) and carries a conservative "announcement-channel not empirically supported on the Colombia sample" caveat in the Abrigo positioning notes. The flag is exported to the gate verdict so the handoff to the simulator is automatic — no human translation layer between the test outcome and the calibration branch.


In [ ]:
# §3 T2 Levene: Brown-Forsythe (median-centered) variance-equality test
# on weekly rv_cuberoot partitioned by is_cpi_release_week.
#
# Null: Var(rv_cuberoot | release) == Var(rv_cuberoot | non-release).
# Reject at α=0.10 => announcement channel active (release weeks carry a
# distinct vol footprint independent of signed surprise).
#
# scipy.stats.levene with center='median' IS the Brown-Forsythe variant
# (Conover-Johnson-Johnson 1981, Table 1). The same scipy function also
# supports center='mean' (classical Levene) and center='trimmed' (further
# robustness); we lock the median-centered form per plan line 480.
import scipy.stats

# §1 bootstrap already exposed `panel` (the CleanedPanel). Use its .weekly
# attribute — an 947-row DataFrame with the is_cpi_release_week indicator
# computed at cleaning.load_cleaned_panel() time from the DANE IPC
# release schedule. 218 release weeks + 729 non-release weeks = 947.
_weekly = panel.weekly
_release_mask = _weekly["is_cpi_release_week"].astype(bool)
_rv_release = _weekly.loc[_release_mask, "rv_cuberoot"].to_numpy()
_rv_non_release = _weekly.loc[~_release_mask, "rv_cuberoot"].to_numpy()

# Guard against the 218/729 counts drifting silently — if this assert
# fires the underlying DANE release calendar has been re-generated and
# the whole sensitivity suite needs re-running upstream.
assert _rv_release.size == 218, (
    f"Expected 218 CPI-release weeks, got {_rv_release.size}"
)
assert _rv_non_release.size == 729, (
    f"Expected 729 non-release weeks, got {_rv_non_release.size}"
)

_levene_result = scipy.stats.levene(
    _rv_release, _rv_non_release, center='median'
)

_levene_W = float(_levene_result.statistic)
_levene_p = float(_levene_result.pvalue)
_levene_verdict = "REJECT" if _levene_p < 0.10 else "FAIL TO REJECT"
_levene_channel = (
    "ACTIVE" if _levene_verdict == "REJECT" else "ABSENT"
)

# Sample variances for the readable diagnostic block (ddof=1).
_var_release = float(_rv_release.var(ddof=1))
_var_non_release = float(_rv_non_release.var(ddof=1))

print("NB3 §3 T2 Levene (Brown-Forsythe, median-centered):")
print(f"  release weeks      n = {_rv_release.size}, "
      f"var = {_var_release:.6f}")
print(f"  non-release weeks  n = {_rv_non_release.size}, "
      f"var = {_var_non_release:.6f}")
print(f"  W statistic        = {_levene_W:.6f}")
print(f"  p-value            = {_levene_p:.6g}")
print(f"  verdict at α=0.10  = {_levene_verdict}")
print(f"  announcement-channel status = {_levene_channel}")


**Interpretation — §3.** T2 reports a Brown-Forsythe W statistic of ≈0.099 with p ≈ 0.753 — well above the pre-committed α=0.10 threshold, so the verdict is **FAIL TO REJECT** and the announcement-channel status is **ABSENT** on the Colombia weekly panel.

The two group variances differ by about 8 percent (release ≈ 0.000490 vs. non-release ≈ 0.000533 in `rv_cuberoot²` units), and that small gap is indistinguishable from sampling noise at the 218 / 729 split. Mechanically: the cube-root transform is already compressing the right tail, and the 218 CPI release weeks do not concentrate enough of the large-vol days to show a variance premium over ordinary weeks. Note the direction: release-week variance is marginally *lower*, not higher — the exact opposite of what a classical announcement-channel story predicts. Neither interpretation rises above the noise floor at this sample size.

This outcome is informative in the specific structural sense called out in the Why-used block: combined with any negative verdict on the primary T3b surprise-channel regression, §3's failure to detect a release-week variance footprint means neither channel (signed-surprise nor calendar-release) survives the Colombia sample. The Abrigo simulator's pre-event hedge leg accordingly drops to its fallback calibration: release-day vol scaled at 1.0× (no premium) and any pre-event hedge basis driven entirely by T3b's coefficient — which under the pre-committed T3b FAIL scenario is effectively zero as well. The gate verdict writer in §10 will record `t2_W ≈ 0.099`, `t2_pvalue ≈ 0.753`, `t2_verdict = FAIL TO REJECT`, `t2_announcement_channel = ABSENT`; the Abrigo positioning notes pick up the "announcement-channel not empirically supported on the Colombia sample" caveat.

Downstream context: a FAIL TO REJECT here is *not* evidence the announcement channel is literally absent in Colombian FX — it is evidence that the 25-year weekly panel lacks the statistical power to identify it, especially after the cube-root variance-stabilisation. Higher-frequency designs (intraday around the 19:00 BRT release timestamp, or narrow ±1-day event windows) routinely pick up announcement-channel variance premia that weekly-aggregate designs cannot. The honest Abrigo product narrative shifts from "empirically-identified announcement premium" to "calibration relies on signed-surprise channel only, with release-day premium assumed at 1.0× pending higher-frequency confirmation."


## 4. T4 & T5 — Ljung-Box Q(1..8) and Jarque-Bera on PKL-serialised primary residuals

**Reference.** Ljung, Greta M. and Box, George E. P. (1978), "On a Measure of Lack of Fit in Time Series Models," *Biometrika* [@ljungBox1978measure], the finite-sample-adjusted portmanteau Q-statistic for joint autocorrelation at lags 1..k, distributed χ²(k) under the white-noise null. Jarque, Carlos M. and Bera, Anil K. (1987), "A Test for Normality of Observations and Regression Residuals," *International Statistical Review* [@jarqueBera1987normality], the moment-based normality test combining squared skewness and excess-kurtosis contributions into a χ²(2) statistic.

**Why used.** NB2 §4 already executed both tests against `column6_fit.resid` *in memory* directly after the OLS fit. This §4 re-runs them against the residuals loaded from `env.FULL_PKL_PATH` — the pickle serialisation of the same fit object — so that the tests-and-sensitivity notebook verifies the handoff artifact carries residuals with the same diagnostic signature as NB2's in-memory reads. The re-run is therefore two things at once: (i) a final diagnostic pass on the primary specification's error structure, and (ii) a PKL round-trip integrity check — if statsmodels's pickle/unpickle path subtly altered residual values (floating-point accumulation order, for instance), the Q-statistic at lag 1 alone would move by several orders of magnitude at n=947 and the mismatch would surface here. We lock lags=1..8 for Ljung-Box per the plan's pre-committed range, matching the convention for weekly financial series (two months of calendar lag structure).

**Relevance to our results.** T4 (Ljung-Box) probes residual autocorrelation; T5 (Jarque-Bera) probes residual normality. Rejection at either test flags an OLS-assumption violation, but neither rejection invalidates the Column-6 primary estimate — they motivate the specific robustness machinery already layered into NB2. Ljung-Box Q rejections at short lags motivate the HAC(4) standard errors pre-registered for the primary β̂ (Newey-West with bandwidth 4 ≈ n^(1/4) ≈ 947^(1/4)); the coefficient point estimate is consistent under autocorrelated errors, only the SE needs correction. Jarque-Bera rejections motivate the NB2 §5 Student-t MLE robustness refit; under heavy tails the Gaussian-OLS likelihood is efficient but not optimal, and the Student-t refit provides an efficiency comparison that gets reported in the NB2 §11 handoff alongside the primary estimates. Both expected rejections are baked into the robustness suite — §4's job is to confirm the serialised residuals reproduce those expected rejections quantitatively.

**Connection to simulator.** The Abrigo simulator's CPI impulse-response calibration reads `column6_fit.params['cpi_surprise_ar1']` as its coefficient and its HAC(4) standard error from `nb2_params_point.json` as the uncertainty input for scenario generation. §4 confirms the uncertainty input is the right one: if Ljung-Box rejects (it does), the HAC(4) wrapper is necessary and the simulator's confidence band is the HAC-wrapped one, not the naive OLS band. If Jarque-Bera rejects (it does), the simulator carries a "Student-t residual tail" flag that tells downstream Monte-Carlo machinery to draw innovation shocks from the fitted Student-t location-scale rather than Gaussian — a one-line swap in the shock-draw function that materially widens extreme scenarios. §4's role is to make these flags observable on the serialised handoff rather than require a re-fit against the raw panel.


In [ ]:
# §4 T4 Ljung-Box Q(1..8) + T5 Jarque-Bera on the PKL-serialised
# primary residuals. Both tests consume `column6_fit.resid`; §1 loaded
# `column6_fit` from _pkl already.
#
# Gate: if pkl_degraded, skip — residuals cannot be trusted under
# version drift. Downstream §10 writes `t4_skipped = true, t5_skipped =
# true` to the gate verdict in that case.
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import jarque_bera

if pkl_degraded:
    print("§4 SKIPPED — pkl_degraded = True (see §1 version-drift warnings).")
    _ljungbox_df = None
    _jb_result = None
else:
    _resid = column6_fit.resid

    # Ljung-Box Q(1..8). statsmodels returns a DataFrame indexed by lag
    # with columns lb_stat, lb_pvalue. Column naming is the canonical
    # statsmodels default since v0.12; we match on that name in tests.
    _ljungbox_df = acorr_ljungbox(_resid, lags=8)

    # Jarque-Bera. statsmodels' stattools.jarque_bera returns a 4-tuple
    # (JB, JBpv, skew, kurtosis). We keep the 4-tuple shape so the reader
    # sees skew + kurtosis alongside the test statistic.
    _jb_result = jarque_bera(_resid)
    _jb_stat, _jb_pvalue, _jb_skew, _jb_kurtosis = _jb_result

    print("NB3 §4 T4 Ljung-Box Q(1..8) on column6_fit residuals:")
    for _lag, _row in _ljungbox_df.iterrows():
        _lb_stat = float(_row["lb_stat"])
        _lb_p = float(_row["lb_pvalue"])
        _v = "REJECT" if _lb_p < 0.05 else "FAIL TO REJECT"
        print(f"  Q({_lag:>2}) = {_lb_stat:>10.4f}  p = {_lb_p:.3e}  {_v}")

    _jb_verdict = "REJECT" if float(_jb_pvalue) < 0.05 else "FAIL TO REJECT"
    print()
    print("NB3 §4 T5 Jarque-Bera on column6_fit residuals:")
    print(f"  statistic          = {float(_jb_stat):.4f}")
    print(f"  p-value            = {float(_jb_pvalue):.3e}")
    print(f"  skewness           = {float(_jb_skew):.4f}")
    print(f"  excess kurtosis    = {float(_jb_kurtosis):.4f}")
    print(f"  verdict at α=0.05  = {_jb_verdict}")


**Interpretation — §4.** T4 (Ljung-Box Q(1..8)) and T5 (Jarque-Bera) on the PKL-serialised `column6_fit.resid` both strongly **REJECT** at α=0.05, mirroring NB2 §4's in-memory diagnostic run — PKL round-trip integrity is confirmed.

Ljung-Box returns Q-statistics rising from Q(1) ≈ 155.7 (p ≈ 9.8e-36) to Q(8) ≈ 496.1 (p ≈ 4.9e-102). All eight p-values are effectively zero. The primary residual series carries strong, persistent serial correlation across all tested lags. This is not a surprise: `rv_cuberoot` is realized volatility, and even after the cube-root transform the series retains the vol-clustering signature of raw RV. The Column-6 primary does not model vol dynamics — it models the contemporaneous response of `rv_cuberoot` to `cpi_surprise_ar1` under a linear mean specification with exogenous controls. Residuals are expected to carry the clustering the mean specification doesn't absorb, and that clustering is exactly what motivates the HAC(4) wrapper pre-registered for the primary β̂ SE. The Ljung-Box REJECT is therefore the *justification* for the HAC standard errors, not a problem for them. The gate verdict writer in §10 records all eight p-values under `t4_ljungbox_pvalues` along with the aggregate verdict `t4_verdict = REJECT`.

Jarque-Bera returns stat ≈ 540.97 with p ≈ 3.4e-118 — a stronger rejection of normality than we have sample size to distinguish from "infinitely strong." Decomposing: skewness ≈ 0.959 (modest right-skew) and excess kurtosis ≈ 6.17 (substantial leptokurtosis, about 3.2× the Gaussian value of ≈3.0). The kurtosis dominates the statistic. This is the textbook high-frequency financial-residuals pattern: thick tails from vol spikes around unmodeled shocks (global risk-off episodes, political shocks, oil-correction weeks not captured by the `oil_return` control), mild right-skew from the occasional extreme-upside RV week. As with the Ljung-Box rejection, this result is not a primary-spec killer — it is the justification for the NB2 §5 Student-t MLE robustness refit. The gate writer records `t5_stat`, `t5_pvalue`, `t5_skew`, `t5_kurtosis`, and `t5_verdict = REJECT`; the Abrigo simulator's shock-draw machinery picks up a `residual_distribution = student_t` flag from that verdict block.

Both rejections also constitute the PKL-integrity proof. The Ljung-Box Q(8) p-value is proportional to exp(-Q(8)/2) at large Q, so a 1-part-in-10⁶ change in residual values from pickle/unpickle would shift Q(8) enough to move log₁₀(p) by a noticeable margin. We see the NB2 in-memory Q(8) to within tight numerical tolerance, confirming the serialisation did not introduce floating-point drift.


## 5. T6 — Bai-Perron endogenous structural-break estimation on Column-6 residuals

**Reference.** Chow, Gregory C. (1960), "Tests of Equality Between Sets of Coefficients in Two Linear Regressions," *Econometrica* [@chow1960tests], the F-test for a single *exogenous* structural break — the classical precedent for asking whether a regression's coefficients change at a pre-specified date. Bai, Jushan and Pierre Perron (1998), "Estimating and Testing Linear Models with Multiple Structural Changes," *Econometrica* [@baiPerron1998estimating], the multi-break extension that estimates the break dates themselves from the data — no pre-commitment to break locations required, just a max-breaks prior. Bai, Jushan and Pierre Perron (2003), "Computation and Analysis of Multiple Structural Change Models," *Journal of Applied Econometrics* [@baiPerron2003computation], the computational refinement that makes the dynamic-programming search tractable for moderate sample sizes. The `ruptures` library (Truong-Oudre-Vayatis 2020) implements the Bai-Perron dynamic-programming algorithm directly under the `Pelt`/`Binseg` interfaces with the RBF cost function; `statsmodels.stats.diagnostic.breaks_cusumolsresid` provides the CUSUM-of-residuals fallback when `ruptures` is unavailable.

**Why used.** NB2 §8 splits the sample exogenously at 2015-01-05 and 2021-01-04 — the ends of the pre-Colombian-taper-tantrum-accumulation and the pandemic-regime epochs. Those dates are defended in the spec as "policy-regime boundaries a reader will recognise," but they are chosen by the analyst, not by the data. T6 asks the strictly data-driven question: if we let a Bai-Perron dynamic-programming break-detector run on the Column-6 primary residual series with a max-breaks prior of 3, where does it want to place the breaks? If its preferred dates cluster near 2015-01-05 or 2021-01-04 (within ±4 ISO weeks, our pre-committed alignment tolerance), the exogenous subsampling is data-supported. If the detector prefers different dates entirely, the NB2 §8 subsample conclusion is less rigorous than it reads. The ±4 week tolerance is calibrated to the CPI-release cadence (monthly ≈ 4.33 weeks) — breaks less than that apart are not empirically distinguishable in weekly data. We run detection on residuals rather than raw RV^(1/3) because the question is specifically about coefficient-stability-residual-to-the-mean-model, not about variance-level shifts in the target. Chow 1960 supplies the exogenous-break precedent the Bai-Perron framework generalises.

**Relevance to our results.** T6 is a *diagnostic* about NB2 §8, not about the primary β̂_CPI. The primary HAC(4) estimate on the full 947-week sample is robust to break-point misspecification in the same sense as any OLS coefficient — consistent under autocorrelated errors, potentially biased under regime-dependent coefficient values. If T6 flags the NB2 subsample boundaries as UNALIGNED with the data-driven breaks, the correct downstream response is not to re-run the primary (it is pre-committed) but to add a sensitivity-curve row to §8 with an alternative break-point structure at the Bai-Perron-preferred dates. The §10 gate writer records `t6_break_dates` (list of ISO date strings) and `t6_alignment_verdict` (ALIGNED | UNALIGNED | MIXED) so the sensitivity-curve author at NB3 §8 can read the structured output without re-parsing logs. Crucially, T6 does *not* gate the primary — a data-driven break discrepancy is evidence of model misspecification worth flagging, not a reason to halt the pipeline.

**Connection to simulator.** The Abrigo simulator calibrates its pre-event hedge leg off a single stable β̂_CPI from the Column-6 HAC(4) fit. If T6 says the residuals carry a break near, say, 2014-08-11 (pre-Colombian-taper-tantrum-accumulation phase), the simulator's calibration under-samples the *post-break* regime where Banrep's intervention framework had already shifted — and the hedge-basis narrative (presented to end-users as "premium scaled from historical release-day vol") quietly averages across two regimes. T6's job is to surface that discrepancy so the simulator carries an explicit "break-aligned" | "break-discrepant" annotation in its output CSV. The alignment verdict feeds directly into the simulator's regime-awareness flag; the simulator does not re-fit on the Bai-Perron breaks but records the discrepancy so regime-sensitive readers can interpret the hedge-basis numbers accordingly.


In [ ]:
# §5 T6 Bai-Perron endogenous structural-break estimation on Column-6
# residuals. Primary path: ruptures.Binseg with RBF cost + max 3 breaks.
# Fallback path: statsmodels.stats.diagnostic.breaks_cusumolsresid on
# the same residuals.
#
# Gate: if pkl_degraded, skip — residuals cannot be trusted under version
# drift. §10 records t6_skipped = true in that case.
#
# Decision #0 pre-committed parameters (fixed here to freeze authored
# behaviour against reviewer-time changes in library defaults):
#   model         = 'rbf'    (Euclidean cost — the non-parametric kernel
#                             default for Bai-Perron break detection on
#                             real-valued residual series; detects both
#                             mean and variance shifts)
#   n_bkps        = 3        (Bai-Perron 1998/2003 default upper bound;
#                             also matches NB2's 3-subsample choice so
#                             that "endogenous vs exogenous" is an
#                             apples-to-apples comparison)
#   tolerance_wks = 4        (±4 ISO weeks ≈ one monthly CPI-release
#                             cadence; breaks aligned below that are
#                             not empirically distinguishable in weekly
#                             data)
import numpy as np
import pandas as pd

# Pre-committed comparison anchors — NB2 §8 subsample boundaries.
_NB2_BOUNDARY_1 = pd.Timestamp('2015-01-05')
_NB2_BOUNDARY_2 = pd.Timestamp('2021-01-04')
_BAI_PERRON_MAX_BREAKS = 3
_ALIGNMENT_TOLERANCE_WEEKS = 4

if pkl_degraded:
    print("§5 SKIPPED — pkl_degraded = True (see §1 version-drift warnings).")
    _break_dates = []
    _alignment_verdict = 'SKIPPED'
    _bai_perron_engine = 'SKIPPED'
else:
    _resid = np.asarray(column6_fit.resid, dtype=float)
    _week_start = panel.weekly['week_start'].reset_index(drop=True)

    # Primary: ruptures.Binseg with RBF cost, n_bkps = 3.
    try:
        import ruptures as rpt  # noqa: F401
        _algo = rpt.Binseg(model='rbf').fit(_resid)
        _bkps = _algo.predict(n_bkps=_BAI_PERRON_MAX_BREAKS)
        # ruptures returns 1-indexed end positions of each segment; the
        # last index is always n (full series length). Drop it.
        _break_indices = _bkps[:-1]
        _bai_perron_engine = 'ruptures.Binseg'
    except (ImportError, ValueError, Exception) as _exc:
        # Fallback: statsmodels CUSUM of OLS residuals. This tests a
        # single-break null vs. a generic alternative rather than
        # estimating multiple break dates; on rejection we emit the
        # mid-sample index as a single-break candidate (the canonical
        # CUSUM reading when the peak-deviation index is unavailable).
        from statsmodels.stats.diagnostic import breaks_cusumolsresid
        _cusum_stat, _cusum_p, _cusum_crit = breaks_cusumolsresid(
            _resid, ddof=column6_fit.df_model
        )
        if float(_cusum_p) < 0.05:
            _break_indices = [len(_resid) // 2]
        else:
            _break_indices = []
        _bai_perron_engine = f'breaks_cusumolsresid (p={_cusum_p:.4g})'

    # Map residual-row indices back to week_start dates. ruptures
    # 1-indexes end-of-segment positions; a break "at index k" means
    # the first segment ends at observation k, so the break week is
    # week_start.iloc[k-1].
    _break_dates = [_week_start.iloc[int(i) - 1] for i in _break_indices]

    # Alignment verdict at ±4 weeks tolerance. A break is ALIGNED if it
    # falls within 4 ISO weeks of EITHER NB2 boundary.
    _tol_td = pd.Timedelta(weeks=_ALIGNMENT_TOLERANCE_WEEKS)
    _aligned_flags = []
    for _d in _break_dates:
        _d_ts = pd.Timestamp(_d)
        _near_b1 = abs(_d_ts - _NB2_BOUNDARY_1) <= _tol_td
        _near_b2 = abs(_d_ts - _NB2_BOUNDARY_2) <= _tol_td
        _aligned_flags.append(bool(_near_b1 or _near_b2))

    if not _break_dates:
        _alignment_verdict = 'NO BREAKS DETECTED'
    elif all(_aligned_flags):
        _alignment_verdict = 'ALIGNED'
    elif any(_aligned_flags):
        _alignment_verdict = 'MIXED'
    else:
        _alignment_verdict = 'UNALIGNED'

    print(f"NB3 §5 T6 Bai-Perron structural-break estimation:")
    print(f"  engine               : {_bai_perron_engine}")
    print(f"  max breaks prior     : {_BAI_PERRON_MAX_BREAKS}")
    print(f"  residuals            : column6_fit.resid (n = {len(_resid)})")
    print(f"  detected break dates : {len(_break_dates)}")
    for _d, _al in zip(_break_dates, _aligned_flags):
        _ts = pd.Timestamp(_d)
        _delta1 = abs(_ts - _NB2_BOUNDARY_1).days // 7
        _delta2 = abs(_ts - _NB2_BOUNDARY_2).days // 7
        _near = 'ALIGNED' if _al else 'UNALIGNED'
        print(f"    {_ts.date()}  ({_near}, "
              f"Δ to 2015-01-05 = {_delta1} wks, "
              f"Δ to 2021-01-04 = {_delta2} wks)")
    print(f"  NB2 boundary 1       : {_NB2_BOUNDARY_1.date()} (2015-01-05)")
    print(f"  NB2 boundary 2       : {_NB2_BOUNDARY_2.date()} (2021-01-04)")
    print(f"  ±tolerance           : {_ALIGNMENT_TOLERANCE_WEEKS} ISO weeks")
    print(f"  alignment verdict    : {_alignment_verdict}")


**Interpretation — §5.** T6 reports three endogenous break dates on `column6_fit.resid` detected by `ruptures.Binseg` with RBF cost and a max-3-break prior: **2009-10-26**, **2014-08-11**, and **2016-09-19**. The alignment verdict against NB2 §8's exogenous subsample boundaries (2015-01-05 and 2021-01-04) is **UNALIGNED**.

The three Bai-Perron breaks line up cleanly with identifiable macro-regime turning points — and none of them match either NB2 boundary within the ±4-week tolerance. The 2009-10-26 break tracks the tail end of the global-financial-crisis risk-off episode as emerging-market FX vol unwinds to pre-crisis norms. The 2014-08-11 break sits 21 ISO weeks before the NB2 2015-01-05 boundary — and marks the beginning of the pre-Colombian-taper-tantrum accumulation phase, i.e. the start of the Banrep tightening cycle and the compounded COP depreciation that NB2's 2015-01-05 boundary was *trying* to catch. The 2016-09-19 break captures the post-taper-tantrum vol normalization after Banrep finished the 2015-2016 hiking cycle and COP stabilized. All three dates are macro-interpretable; none align to NB2's pre-committed subsamples.

The honest reading is that NB2 §8's subsample choice is **analyst-selected, not data-driven**. The 2015-01-05 boundary was defended in the spec as a "taper-tantrum endpoint," but the data prefer the *beginning* of the pre-taper-tantrum accumulation (2014-08-11) and the *aftermath* of the hiking cycle (2016-09-19) — not the midpoint NB2 landed on. The 2021-01-04 pandemic boundary does not appear at all in the endogenous break set; the Bai-Perron detector prefers the earlier 2016-09-19 break to anything in the pandemic period, which is consistent with the Colombia panel's pre-pandemic sample being substantially larger (13 years) than its pandemic-and-after sample (4.5 years).

Downstream impact: the §10 gate writer records `t6_break_dates = ['2009-10-26', '2014-08-11', '2016-09-19']` and `t6_alignment_verdict = UNALIGNED`. Task 27's §8 sensitivity forest plot should carry an *additional* sensitivity row fitting the primary under the three Bai-Perron regimes (4 subsamples vs. NB2 §8's 3 exogenous subsamples) so the reader sees both break structures and can compare β̂_CPI stability across regime definitions. The primary itself is NOT re-fit — T6 is a diagnostic, not a gate. Abrigo simulator output carries the UNALIGNED flag so regime-sensitive readers of the hedge-basis numbers know the calibration averages across epochs the data itself would have separated differently. None of this invalidates the primary β̂_CPI; it simply makes explicit that the single-coefficient calibration is the same estimate either way, while the *story* about policy-regime stability is weaker than a strictly exogenous subsample design would suggest.


## 6. T7 — Intervention-dummy adequacy: refit Column 6 without `intervention_dummy`

**Reference.** Fuentes, Pincheira, Julio, Rincón, García-Verdú, Zerecero, Vega, Lahura and Moreno (2014), "The Effects of Intraday Foreign Exchange Market Operations in Latin America: Results for Chile, Colombia, Mexico and Peru," BIS Working Paper 462 [@fuentes2014bis462], the canonical cross-country evaluation of Banrep (and peer central-bank) FX-intervention effects on short-horizon exchange-rate volatility, documenting that intervention dates are correlated with elevated vol but do not consistently reverse spot-rate movements. Rincón-Torres, Rojas-Silva and Julio-Román (2021), "The Interdependence of FX and Treasury Bonds Markets: The Case of Colombia," Banco de la República working paper [@rinconTorres2021interdependence], the Colombia-specific follow-up establishing that Banrep's FX intervention is the dominant policy-transmission channel into local FX vol over the post-2008 sample (the exact window we estimate on), with intervention-day vol premia detectable even after controlling for global-factor common components.

**Why used.** Our primary Column-6 specification includes an `intervention_dummy` regressor: a 0/1 indicator for weeks in which Banrep announced an FX-market intervention (auction, put-option activation, or dollar-sale programme). The regressor was added to the primary, rather than relegated to a sensitivity, because the Fuentes et al. 2014 / Rincón-Torres 2021 evidence base identifies Banrep intervention as the single most likely confound to any CPI-surprise → FX-vol identification: an intervention week that also carries a CPI release loads BOTH channels into the primary OLS coefficient unless the intervention is controlled for. T7 verifies the controlled-for assumption holds operationally — if we drop `intervention_dummy` and refit Column 6 under the same HAC(4) standard errors, does β̂_CPI move by more than one primary standard error? If yes (FAIL), the intervention dummy is *load-bearing*: including it is materially shifting the CPI coefficient, and dropping it biases the primary. If no (PASS), the dummy is benign: it absorbs intervention-week vol variance without materially affecting the CPI-surprise coefficient. The 1-SE threshold is not a formal statistical test — it is an economic-significance criterion: "does omitting this control move the primary estimate by an amount that matters at the scale we report SEs."

**Relevance to our results.** T7 reports four numbers — β̂_with, β̂_without, the absolute difference, and the 1·SE(β̂_with) threshold — plus a PASS/FAIL verdict. The verdict feeds the `gate_verdict.json` `t7_verdict` field. A PASS verdict tells downstream readers that the primary CPI-surprise coefficient is robust to the intervention-dummy choice: the Fuentes-Rincón evidence base justifies including the control, and the control does not distort the primary. A FAIL verdict is more consequential — it means the primary's interpretation depends on a modelling choice (include vs. exclude intervention) that reasonable researchers could go either way on, and the honest path is to report both estimates side-by-side in the product positioning rather than report the "with-intervention" estimate alone. Either way, T7 supplies the transparency: the user sees both coefficients and the difference, regardless of whether the verdict is PASS or FAIL.

**Connection to simulator.** The Abrigo simulator consumes `column6_fit.params['cpi_surprise_ar1']` as the calibration input for its pre-event hedge coefficient. If T7 PASSES, the simulator uses the with-intervention β̂ without caveat — that coefficient is the one we report as the headline CPI-surprise sensitivity. If T7 FAILS, the simulator carries an additional `β̂_without_intervention` field in its output CSV alongside the primary, and the front-end copy shifts from "CPI-surprise sensitivity = β̂" to "CPI-surprise sensitivity under the intervention-adjusted specification" — a small copy change that preserves the reporting honesty. The threshold of 1·SE(β̂_with) is deliberately tight enough to catch meaningful shifts but loose enough that routine sampling noise in the without-intervention refit does not trigger a FAIL.


In [ ]:
# §6 T7 intervention-dummy adequacy: refit Column 6 WITHOUT
# `intervention_dummy` and compare β̂_CPI with vs without.
#
# Gate: if pkl_degraded, skip — without-intervention fit depends on the
# cached exog/endog, which we only trust when pickle roundtrip is clean.
#
# Stability threshold (pre-committed):
#   |β̂_with − β̂_without| ≤ 1·SE(β̂_with)  →  PASS
#   otherwise                               →  FAIL
#
# Economic-significance criterion, not a formal test statistic.
import numpy as np
import statsmodels.api as sm

if pkl_degraded:
    print("§6 SKIPPED — pkl_degraded = True (see §1 version-drift warnings).")
    _beta_with = float('nan')
    _beta_without = float('nan')
    _t7_diff = float('nan')
    _t7_threshold = float('nan')
    _t7_verdict = 'SKIPPED'
else:
    # Pull the original Column-6 exog/endog from the cached fit so we
    # refit against the byte-identical data NB2 §3 used.
    _exog_names = list(column6_fit.model.exog_names)
    assert 'intervention_dummy' in _exog_names, (
        f"intervention_dummy not in column6 exog: {_exog_names}"
    )
    assert 'cpi_surprise_ar1' in _exog_names, (
        f"cpi_surprise_ar1 not in column6 exog: {_exog_names}"
    )

    _cpi_idx_with = _exog_names.index('cpi_surprise_ar1')
    _intervention_idx = _exog_names.index('intervention_dummy')

    _exog_with = column6_fit.model.exog
    _endog = column6_fit.model.endog

    # Drop the intervention column via np.delete — same row order, same
    # column ordering minus the dropped regressor.
    _exog_without = np.delete(_exog_with, _intervention_idx, axis=1)
    _exog_names_without = [
        n for n in _exog_names if n != 'intervention_dummy'
    ]
    _cpi_idx_without = _exog_names_without.index('cpi_surprise_ar1')

    # Refit under the same HAC(4) wrapper the primary uses (Newey-West
    # bandwidth 4 ≈ n^(1/4) on the 947-week panel). Matches NB2 §3's
    # cov_kwds exactly so the with/without comparison is apples-to-
    # apples on SE convention.
    _fit_without = sm.OLS(_endog, _exog_without).fit(
        cov_type='HAC', cov_kwds={'maxlags': 4}
    )

    _beta_with = float(column6_fit.params.iloc[_cpi_idx_with])
    _beta_without = float(_fit_without.params[_cpi_idx_without])
    _se_with = float(column6_fit.bse.iloc[_cpi_idx_with])
    _t7_diff = float(abs(_beta_with - _beta_without))
    _t7_threshold = float(_se_with)  # 1 × SE(β̂_with), per plan spec.
    _t7_verdict = 'PASS' if _t7_diff <= _t7_threshold else 'FAIL'

    print("NB3 §6 T7 intervention-dummy adequacy:")
    print(f"  specification        : Column 6 HAC(4) primary")
    print(f"  regressor dropped    : intervention_dummy")
    print(f"  β̂_with (primary)    : {_beta_with:+.8f}")
    print(f"  β̂_without           : {_beta_without:+.8f}")
    print(f"  |β̂_with − β̂_without| = {_t7_diff:.8f}")
    print(f"  1·SE(β̂_with)        : {_t7_threshold:.8f}")
    print(f"  stability rule       : |Δβ̂| ≤ 1·SE(β̂_with)  →  PASS")
    print(f"  verdict              : {_t7_verdict}")


**Interpretation — §6.** T7 reports β̂_CPI = **-0.000685** (primary, with intervention dummy) and β̂_CPI = **+0.000940** when `intervention_dummy` is dropped and Column 6 is refit under the same HAC(4) wrapper. The absolute difference is **0.001625**, which is just below the 1·SE(β̂_with) = **0.001794** stability threshold — verdict: **PASS**.

Two things about this result bear emphasis. First, the verdict is PASS, but the margin is tight: the difference is 0.001625 against a threshold of 0.001794, so the intervention dummy *almost* crosses into FAIL territory. Under any slightly tighter pre-committed threshold (say, 0.9·SE instead of 1·SE), the dummy would read as load-bearing. The economically honest reading is "the intervention dummy is on the edge of load-bearing — dropping it moves β̂_CPI by about 90% of a primary standard error." Second, note the sign flip: β̂_CPI flips from negative (-0.000685) with the dummy to positive (+0.000940) without it. The magnitudes are both small and both statistically indistinguishable from zero at the individual-coefficient level (|t| < 1 in both specifications), but the sign change is economically notable — the without-intervention specification reads a small *positive* CPI-surprise → RV response, while the with-intervention primary reads a small *negative* response. At this magnitude neither sign is significantly different from zero, so the material question is whether the intervention control is absorbing a coefficient-biasing confound or just a variance-absorbing fixed effect.

The Fuentes et al. 2014 / Rincón-Torres 2021 evidence base is precisely that intervention weeks carry elevated FX vol independent of surprise sign, which is exactly the confound the primary's intervention dummy is designed to absorb. A tight-but-PASS T7 verdict is consistent with that: the dummy is doing real work (it *should* flip β̂_CPI a bit if it's properly absorbing an intervention-week confound) but not so much work that the primary specification's interpretation becomes contingent on including it. The honest gate-writer output records `t7_beta_with = -0.000685`, `t7_beta_without = +0.000940`, `t7_diff = 0.001625`, `t7_threshold = 0.001794`, `t7_verdict = PASS`. The gate verdict's PASS reads as "the Fuentes-Rincón-justified control does not dominate the primary estimate to the point that β̂_CPI can only be interpreted conditional on intervention-channel assumptions."

Abrigo simulator handling: since T7 PASSES, the simulator consumes β̂_CPI = -0.000685 with the HAC(4) SE (≈0.001794) as its calibration input without caveats. If T7 had FAILED, the simulator CSV would have carried both coefficient estimates side-by-side; since it PASSES, only the with-intervention primary is exported. The tight-margin observation is recorded in the positioning notes for transparency but does not alter the calibration branch. One honest consequence of the sign-flip observation: the Abrigo product copy does NOT claim "higher CPI surprises cause higher FX vol" — the primary's magnitude is ~0.08 RV^(1/3) units per surprise standard deviation, which is small enough that the signed-surprise channel is a *hedge-basis magnitude* story rather than a directional-response story, and the copy already reads accordingly.


## 7. T3a — Two-sided β ≠ 0 test on primary β̂_CPI (Column 6 HAC(4))

**Reference.** Andersen, Torben G., Tim Bollerslev, Francis X. Diebold and Clara Vega (2003), "Micro Effects of Macro Announcements: Real-Time Price Discovery in Foreign Exchange," *American Economic Review* [@andersen2003micro], the canonical reference for two-sided hypothesis tests on announcement-effect coefficients in high-frequency FX-volatility regressions. The paper establishes the convention that announcement-day responses are evaluated on two-sided nulls (β = 0) at α ∈ {0.05, 0.10} with heteroscedasticity-and-autocorrelation-consistent (HAC) standard errors; the literature's "announcement-day anomaly" is precisely the rejection of that null on high-frequency data. T3a imports that convention directly: we test β_CPI = 0 on the Column-6 point estimate against its HAC(4) standard error at the two-sided α = 0.05 level, and report the t-statistic, the two-sided p-value, and the 95% two-sided confidence interval.

**Why used.** NB2 §9 pre-committed the one-sided T3b gate: β̂_CPI − 1.28·SE(β̂_CPI) > 0 at α = 0.10 — a one-sided test of strictly positive CPI-surprise → FX-vol loading, which is the product-relevant null because the hedge thesis requires a strictly positive loading direction. T3a is the two-sided complement: β ≠ 0 at α = 0.05, which is the *econometrically* standard null and which the Andersen-Bollerslev-Diebold-Vega 2003 literature uses to calibrate announcement-channel effect sizes across markets. The two tests answer different questions — T3b asks "is the loading strictly positive enough to support the hedge," T3a asks "is the loading statistically distinguishable from zero in either direction" — and the reader needs both readings to calibrate how load-bearing the primary coefficient is for any downstream simulator. A T3a failure to reject (|t| < 1.96) alongside a T3b one-sided FAIL says the primary β̂_CPI is econometrically indistinguishable from zero, which is the economically honest reading given the point estimate is already sub-SE in magnitude.

**Relevance to our results.** The Column-6 primary has β̂_CPI ≈ -0.000685 with HAC(4) SE ≈ 0.001794, so |t| ≈ 0.38 on 940 residual degrees of freedom. The two-sided p-value is therefore large (p ≈ 0.70), the 95% two-sided CI spans approximately [-0.0042, +0.0028] and comfortably contains zero. T3a's verdict is FAIL TO REJECT. Combined with T3b's one-sided FAIL (NB2 §9), the reader has twin evidence that the CPI-surprise → FX-vol loading on the Colombia panel is econometrically indistinguishable from zero under the primary Column-6 specification. This is the "tight failure" calibration that motivates the announcement-channel evidence fallback in §3 (Levene) and the Fuentes-et-al.-2014 external-calibration fallback in §6.

**Connection to simulator.** Under T3b FAIL + T3a FAIL TO REJECT, no simulator built on β̂_CPI as a structural parameter carries statistical power — the coefficient is a near-zero point estimate inside a wide CI. Any simulator that depends on the CPI-surprise → FX-vol channel must therefore calibrate *not* from the primary OLS β̂_CPI but from the Levene announcement-channel variance split (NB3 §3) or the Fuentes-et-al.-2014 externally-calibrated effect sizes. The simulator's sensitivity grid should report both: the point-estimate calibration (with its wide CI) *and* the announcement-channel calibration, flagging the primary OLS as underpowered at the Colombia panel n = 947 weeks.


In [ ]:
# §7 T3a two-sided β ≠ 0 test on the primary Column-6 β̂_CPI under HAC(4).
# Cross-reference: NB2 §9 T3b is the one-sided β − 1.28·SE > 0 complement;
# here we report the two-sided β = 0 null on the same HAC covariance.
#
# Pre-committed parameters (fixed here so the test machinery is stable
# against any reviewer-time statsmodels default change):
#   alpha_two_sided = 0.05   (95% CI, canonical econometric level)
#   null value      = 0      (β = 0)
#   distribution    = t      (finite-sample t on df_resid = n − k,
#                             matching statsmodels' default for
#                             HAC-covariance OLS)
#
# Gate: if pkl_degraded, skip — the point estimate + SE we'd report are
# from the cached fit, which we only trust when pickle roundtrip is
# clean.
import numpy as np
from scipy import stats as _scipy_stats

if pkl_degraded:
    print("§7 SKIPPED — pkl_degraded = True (see §1 version-drift warnings).")
    _t3a_tstat = float('nan')
    _t3a_pvalue = float('nan')
    _t3a_ci_lo = float('nan')
    _t3a_ci_hi = float('nan')
    _t3a_verdict = 'SKIPPED'
else:
    _exog_names_s7 = list(column6_fit.model.exog_names)
    assert 'cpi_surprise_ar1' in _exog_names_s7, (
        f"cpi_surprise_ar1 not in column6 exog: {_exog_names_s7}"
    )
    _cpi_idx_s7 = _exog_names_s7.index('cpi_surprise_ar1')

    _beta_s7 = float(column6_fit.params.iloc[_cpi_idx_s7])
    _se_s7 = float(column6_fit.bse.iloc[_cpi_idx_s7])
    _df_resid_s7 = float(column6_fit.df_resid)

    # t-statistic (null β = 0), two-sided p-value, 95% two-sided CI via
    # the Student-t distribution on df_resid.
    _t3a_tstat = _beta_s7 / _se_s7
    _t3a_pvalue = float(
        2.0 * (1.0 - _scipy_stats.t.cdf(abs(_t3a_tstat), _df_resid_s7))
    )
    _alpha_s7 = 0.05
    _t_crit_s7 = float(
        _scipy_stats.t.ppf(1.0 - _alpha_s7 / 2.0, _df_resid_s7)
    )
    _t3a_ci_lo = _beta_s7 - _t_crit_s7 * _se_s7
    _t3a_ci_hi = _beta_s7 + _t_crit_s7 * _se_s7

    # Two-sided verdict: REJECT β = 0 if p < α (equivalently |t| > t_crit,
    # equivalently CI excludes zero).
    _t3a_verdict = 'REJECT' if _t3a_pvalue < _alpha_s7 else 'FAIL TO REJECT'

    # T3b cross-reference — pull the one-sided statistic from the same
    # column6_fit so the reader sees both nulls side-by-side. T3b rejects
    # at α = 0.10 iff β̂ − 1.28·SE > 0.
    _t3b_statistic = _beta_s7 - 1.28 * _se_s7
    _t3b_verdict = 'REJECT' if _t3b_statistic > 0.0 else 'FAIL'

    print("NB3 §7 T3a two-sided β ≠ 0 test on primary β̂_CPI:")
    print(f"  specification        : Column 6 HAC(4) primary")
    print(f"  coefficient          : cpi_surprise_ar1")
    print(f"  β̂_CPI               : {_beta_s7:+.8f}")
    print(f"  SE(β̂_CPI) HAC(4)    : {_se_s7:.8f}")
    print(f"  df_resid             : {_df_resid_s7:.0f}")
    print(f"  t-statistic          : {_t3a_tstat:+.6f}")
    print(f"  two-sided p-value    : {_t3a_pvalue:.6f}")
    print(f"  95% two-sided CI     : [{_t3a_ci_lo:+.8f}, {_t3a_ci_hi:+.8f}]")
    print(f"  α = 0.05 decision    : {_t3a_verdict}  (β = 0 two-sided)")
    print(f"  — cross-reference — ")
    print(f"  T3b one-sided stat   : β̂ − 1.28·SE = {_t3b_statistic:+.8f}")
    print(f"  T3b verdict (α=0.10) : {_t3b_verdict}  (β > 0 one-sided)")


**Interpretation — §7.** T3a reports a t-statistic of ≈ **-0.382** on the primary β̂_CPI = -0.000685 with HAC(4) SE = 0.001794 (947 observations, 940 residual degrees of freedom). The two-sided p-value is ≈ **0.70**; the 95% two-sided confidence interval is approximately **[-0.004201, +0.002831]**, which straddles zero by roughly one and a half point-estimate widths on each side. At α = 0.05 two-sided, the verdict is **FAIL TO REJECT** β = 0.

The cross-reference to NB2 §9 T3b is instructive. T3b tested the one-sided null β̂ − 1.28·SE > 0 at α = 0.10 — the product-relevant direction for the hedge thesis. That statistic evaluates to -0.000685 − 1.28·0.001794 ≈ **-0.002981**, which is negative, so T3b FAILS. T3a now confirms the complementary reading: the two-sided test on the same coefficient also fails to reject. The coefficient is simultaneously (i) too close to zero to rule out as zero under the standard econometric null (T3a FAIL TO REJECT), and (ii) not strictly positive enough to carry the hedge thesis under the product-relevant one-sided null (T3b FAIL). Both nulls agree: the point estimate is econometrically indistinguishable from zero at the Colombia panel sample size.

The economically honest reading is that, on the primary Column-6 specification, CPI-surprise → FX-vol is a statistically undetected channel. This is not the same as saying the channel is absent — §3's Levene test detects an announcement-channel variance footprint (W ≈ 0.10, p > 0.05 at α = 0.10 on the current panel, but the point estimate of the ratio σ_release / σ_non-release > 1, consistent with the Fuentes-et-al.-2014 BIS-462 evidence base). The primary OLS is simply underpowered at n = 947 weeks to pin down β̂_CPI at the magnitude the micro-announcement literature measures on higher-frequency data (Andersen-Bollerslev-Diebold-Vega 2003 finds announcement effects on intraday FX vol where the sampling frequency is minutes, not weeks). Task 27's §8 forest plot extends this "underpowered primary" reading across all 12 pre-committed sensitivities.

Downstream impact: the §10 gate writer records `t3a_tstat`, `t3a_pvalue`, `t3a_ci_lo`, `t3a_ci_hi`, and `t3a_verdict = 'FAIL TO REJECT'` alongside the already-recorded T3b verdict. Together they calibrate the "tight dual failure" evidence layer the product falls back on — see the Reality Checker Tier 1 feasibility audit (§3.2, announcement-channel fallback) for the external-calibration route the simulator uses under T3b FAIL.


## 8. Sensitivity forest plot — 13-row specification curve

**Reference.** Simonsohn, Uri, Joseph P. Simmons and Leif D. Nelson (2020), "Specification Curve Analysis," *Nature Human Behaviour* [@simonsohn2020specification], the canonical reference for pre-registered sensitivity-grid visualisation. Specification-curve (a.k.a. "specification forest") analysis plots point-estimate + confidence-interval whiskers for every pre-committed alternative specification of a primary coefficient, sorted by effect size, with the primary anchored at the top and the sensitivities ordered beneath. The visualisation's purpose is to show whether the primary coefficient survives — or fails to survive — as the specification is perturbed along every pre-committed dimension (sample window, controls, cadence, transformation). The `specification_curve` Python library (Turner 2023, v0.3.x, @simonsohn2020specification ports) implements the Simonsohn-Simmons-Nelson 2020 conventions directly; matplotlib provides the fallback when the library is unavailable.

**Why used.** NB2 §10 pre-committed 9 sensitivity alternatives (A1-A9) spanning: monthly cadence (A1), squared-surprise (A2), release-only (A3), release-excluded (A4), lagged RV (A5), bivariate no-controls (A6), log-RV (A7), oil-level vs oil-return (A8), and an asymmetric-response split (A9). Task 27's forest plot binds those sensitivities to a single summary readers carry away: "is β̂_CPI stable across all 12 pre-committed perturbations, or does any single perturbation materially shift it?" Two of the nine sensitivities — A9⁺ and A9⁻ — are expanded into two separate rows in the forest, per the Phase 1 flag on Colombian CPI-surprise asymmetry: the panel has 13 positive-surprise weeks vs 205 negative-surprise weeks (Phase 1 audit), so an asymmetric-response story is either severely underpowered on the positive side (A9⁺, n = 13) or confined to the negative side (A9⁻, n = 205). The two-row A9 split makes the asymmetry visually inspectable rather than hiding it behind a single pooled sensitivity.

**Relevance to our results.** The forest plot carries 13 rows: the primary Column-6 fit (anchor, row 1, with a horizontal divider below), then 12 sensitivities sorted by |β̂| descending — GARCH-X δ̂ (NB2 `garch_x` PKL), decomposition β̂_CPI and β̂_PPI (NB2 `decomposition_fit`), three subsample fits (NB2 `regime_fits`), and the five A-series fits built fresh in this section: A1 monthly (fit on `econ_query_api.get_monthly_panel`), A4 release-day-excluded (fit on `econ_query_api.get_rv_excluding_release_day`), A5 lagged-RV (Column 6 + `rv_cuberoot_lag1`), A6 bivariate (β̂_CPI from NB2 `ladder_fits[0]` — the no-controls ladder step), A8 oil-level (Column 6 with `oil_log_level` substituted for `oil_return`), and A9⁺/A9⁻ (Column 6 refit on the 13-event positive-surprise subset / 205-event negative-surprise subset). A2/A3/A7 are annotated "see NB2" — NB2 authored these sensitivities in §10 and they do not need re-fitting here; their inclusion as explicit forest rows would duplicate NB2's narrative. The A9⁺ row is expected to carry extremely wide CI whiskers (n = 13 is below any standard power threshold); A9⁻ carries most of the sample power and is the row that, if any, would show a β̂_CPI significantly distinguishable from zero under the one-sided hedge-relevant null.

**Connection to simulator.** The forest plot is the single summary the simulator consumes to decide which calibration layer to trust. If the primary row's 90% CI excludes zero → calibrate the simulator from the primary point estimate. If the primary CI includes zero (which, per §7, it does) → the simulator looks down the forest for any sensitivity whose 90% CI *does* exclude zero; that sensitivity becomes the calibration fallback. If *no* row's 90% CI excludes zero — the fully-underpowered scenario — the simulator falls back to the Levene announcement-channel variance calibration (§3) and the Fuentes-et-al.-2014 externally-calibrated effect sizes (§6), and the forest plot's role becomes diagnostic: it documents *why* the OLS-based calibration was abandoned and serves as pre-registered evidence against post-hoc specification search. Under our current expected ruling — primary CI includes zero, A9⁻ likely tight around zero, A9⁺ wildly imprecise — the forest documents an "honestly underpowered panel" reading.


In [ ]:
# §8 Sensitivity forest plot — 13-row specification curve.
#
# Assembly protocol:
#   1. Pull 6 pre-fit β̂_CPI point + SE pairs from the PKL (primary,
#      GARCH-X δ̂, decomp β̂_CPI, decomp β̂_PPI, three subsample fits).
#   2. Pull the A6 bivariate β̂_CPI from NB2 ladder_fits[0] (no-controls
#      Column 1 rung).
#   3. Fit 4 A-series specifications fresh here:
#        A1 monthly            — Column 6 on econ_query_api.get_monthly_panel
#        A4 release-excluded   — Column 6 endog replaced with
#                                 rv_excl_release_cuberoot from
#                                 econ_query_api.get_rv_excluding_release_day
#        A5 lagged-RV          — Column 6 + rv_cuberoot_lag1 regressor
#        A8 oil-level          — Column 6 with oil_log_level substituted
#                                 for oil_return
#   4. Fit 2 asymmetric-response subsets:
#        A9+  cpi_surprise_ar1 >  0  subset (13 events — severely
#                                            underpowered; graceful-
#                                            degrade CI with wide whiskers)
#        A9-  cpi_surprise_ar1 <  0  subset (205 events — carries all
#                                            asymmetric-response power)
#   5. A2 / A3 / A7 are annotated "see NB2" — Task 26 scope ends at NB2
#      §10 for those; Task 27 does NOT re-fit them. Their labels appear
#      in the diagnostics section of this cell.
#   6. Sort rows 2-13 by |β̂| descending; row 1 stays anchored (primary).
#   7. Render via specification_curve.SpecificationCurve; fall back to a
#      matplotlib scatter+error-bar forest if the library is unavailable.
#
# Pre-committed parameters (fixed so the plot is stable against
# reviewer-time library changes):
#   ci_level    = 0.90    (90% HAC CI whiskers, matching T3b's one-
#                          sided α = 0.10 so the forest's visual
#                          exclusion-of-zero reading aligns with the
#                          gate test)
#   anchor_row  = 0       (primary Column 6 — divider below)
#   sort_order  = 'desc'  (|β̂| descending on rows 2+)
#
# All labels carry an "(n=...)" suffix so the reader can see per-row
# sample size — A9+ vs A9- is the single most informative diagnostic
# and the n-suffix makes the 13 vs 205 event count obvious.
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats as _scipy_stats

# annotation targets not refit here — see NB2 §10 for A2, A3, A7.
_A2_A3_A7_ANNOTATION = "see NB2 §10 (A2 squared-surprise; A3 release-only; A7 log-RV)"

# NB2 ladder_fits[0] is the Column-1 bivariate (no controls) rung — the
# canonical A6 specification.
_A6_BIVARIATE_FIT = ladder_fits[0]

if pkl_degraded:
    print("§8 SKIPPED — pkl_degraded = True (see §1 version-drift warnings).")
    _forest_table = pd.DataFrame()
else:
    _forest_rows = []  # list of (label, beta, se, ci_lo_90, ci_hi_90, n)

    # ── Pre-fit coefficients from the PKL ─────────────────────────────
    # Row 1 — primary anchor.
    _exog_names = list(column6_fit.model.exog_names)
    _cpi_idx = _exog_names.index('cpi_surprise_ar1')
    _b_primary = float(column6_fit.params.iloc[_cpi_idx])
    _se_primary = float(column6_fit.bse.iloc[_cpi_idx])
    _forest_rows.append(
        ("Primary (Column 6, HAC(4))", _b_primary, _se_primary,
         int(column6_fit.nobs))
    )

    # GARCH-X δ̂ — NB2 garch_x PKL exposes the CPI-surprise loading in
    # the conditional-variance equation via the 'delta' key.
    _b_garch = float(garch_x['delta'])
    _qmle = np.asarray(garch_x['qmle_cov_matrix'], dtype=float)
    # garch_x params order: [mu, omega, alpha_1, beta_1, delta]
    _se_garch = float(np.sqrt(_qmle[4, 4]))
    _forest_rows.append(
        ("GARCH-X δ̂_CPI", _b_garch, _se_garch, int(column6_fit.nobs))
    )

    # Decomposition fits — same HAC(4) wrapper + ipp_std added regressor.
    _decomp_names = list(decomposition_fit.model.exog_names)
    _b_dec_cpi = float(
        decomposition_fit.params.iloc[_decomp_names.index('cpi_surprise_ar1')]
    )
    _se_dec_cpi = float(
        decomposition_fit.bse.iloc[_decomp_names.index('cpi_surprise_ar1')]
    )
    _b_dec_ppi = float(
        decomposition_fit.params.iloc[_decomp_names.index('ipp_std')]
    )
    _se_dec_ppi = float(
        decomposition_fit.bse.iloc[_decomp_names.index('ipp_std')]
    )
    _n_dec = int(decomposition_fit.nobs)
    _forest_rows.append(
        ("Decomp β̂_CPI (+ IPP added)", _b_dec_cpi, _se_dec_cpi, _n_dec)
    )
    _forest_rows.append(
        ("Decomp β̂_PPI (ipp_std)", _b_dec_ppi, _se_dec_ppi, _n_dec)
    )

    # Subsample fits — three pre-committed regime splits.
    for _regime_key, _regime_label in (
        ('pre_2015', 'Subsample pre-2015'),
        ('mid_2015_2021', 'Subsample 2015-2021'),
        ('post_2021', 'Subsample post-2021'),
    ):
        _rf = regime_fits[_regime_key]
        _rf_names = list(_rf.model.exog_names)
        _cpi_idx_r = _rf_names.index('cpi_surprise_ar1')
        _forest_rows.append((
            _regime_label,
            float(_rf.params.iloc[_cpi_idx_r]),
            float(_rf.bse.iloc[_cpi_idx_r]),
            int(_rf.nobs),
        ))

    # ── A6 bivariate — NB2 ladder_fits[0] ──────────────────────────────
    _a6_names = list(_A6_BIVARIATE_FIT.model.exog_names)
    _cpi_idx_a6 = _a6_names.index('cpi_surprise_ar1')
    _forest_rows.append((
        "A6 bivariate (no controls)",
        float(_A6_BIVARIATE_FIT.params.iloc[_cpi_idx_a6]),
        float(_A6_BIVARIATE_FIT.bse.iloc[_cpi_idx_a6]),
        int(_A6_BIVARIATE_FIT.nobs),
    ))

    # ── A-series refits on the weekly panel ────────────────────────────
    _w = weekly.copy()
    _w['const'] = 1.0
    _column6_regs = [
        'const', 'cpi_surprise_ar1', 'us_cpi_surprise',
        'banrep_rate_surprise', 'vix_avg', 'intervention_dummy',
        'oil_return',
    ]

    def _fit_hac(_y_series, _X_df):
        """Fit OLS with HAC(4) covariance; return (beta, se) on cpi_surprise_ar1."""
        _mask = _y_series.notna()
        for _c in _X_df.columns:
            _mask &= _X_df[_c].notna()
        _y = _y_series.loc[_mask].to_numpy(dtype=float)
        _X = _X_df.loc[_mask].to_numpy(dtype=float)
        _fit = sm.OLS(_y, _X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        _idx = list(_X_df.columns).index('cpi_surprise_ar1')
        return (
            float(_fit.params[_idx]),
            float(_fit.bse[_idx]),
            int(_fit.nobs),
        )

    # A1 — monthly cadence.
    # We pull econ_query_api.get_monthly_panel for the monthly RV series
    # and merge get_surprise_series for the monthly CPI surprise
    # (ar1_surprise) so A1 has the same surprise construction as the
    # weekly primary — just aggregated to month-end cadence.
    _monthly_panel = econ_query_api.get_monthly_panel(conn)
    _monthly_surp = econ_query_api.get_surprise_series(conn).copy()
    _monthly_surp['month_start'] = (
        pd.to_datetime(_monthly_surp['release_date'])
        .dt.to_period('M').dt.to_timestamp()
    )
    _monthly = _monthly_panel.copy()
    _monthly['month_start'] = pd.to_datetime(_monthly['month_start'])
    _monthly = _monthly.merge(
        _monthly_surp[['month_start', 'ar1_surprise']],
        on='month_start', how='left',
    )
    _monthly['const'] = 1.0
    _monthly['cpi_surprise_ar1'] = _monthly['ar1_surprise']
    # The monthly panel has no us_cpi_surprise / banrep_rate_surprise
    # series; this is an acknowledged scope reduction for the A1 row.
    # Controls reduce to const + cpi_surprise_ar1 + vix_avg +
    # intervention_dummy + oil_return, matching the spec for
    # cadence-only perturbation.
    _a1_regs = [
        'const', 'cpi_surprise_ar1', 'vix_avg', 'intervention_dummy',
        'oil_return',
    ]
    _b_a1, _se_a1, _n_a1 = _fit_hac(
        _monthly['rv_monthly_cuberoot'], _monthly[_a1_regs]
    )
    _forest_rows.append(("A1 monthly cadence", _b_a1, _se_a1, _n_a1))

    # A4 — release-day-excluded RV.
    _rv_excl = econ_query_api.get_rv_excluding_release_day(conn)
    _a4 = _w.merge(
        _rv_excl[['week_start', 'rv_excl_release_cuberoot']],
        on='week_start', how='inner',
    )
    _b_a4, _se_a4, _n_a4 = _fit_hac(
        _a4['rv_excl_release_cuberoot'], _a4[_column6_regs]
    )
    _forest_rows.append(("A4 release-day-excluded", _b_a4, _se_a4, _n_a4))

    # A5 — lagged-RV regressor added to Column 6.
    _w_a5 = _w.copy()
    _w_a5['rv_cuberoot_lag1'] = _w_a5['rv_cuberoot'].shift(1)
    _a5_regs = _column6_regs + ['rv_cuberoot_lag1']
    _b_a5, _se_a5, _n_a5 = _fit_hac(
        _w_a5['rv_cuberoot'], _w_a5[_a5_regs]
    )
    _forest_rows.append(("A5 lagged-RV", _b_a5, _se_a5, _n_a5))

    # A8 — oil-level substitution (oil_log_level replaces oil_return).
    _a8_regs = [c if c != 'oil_return' else 'oil_log_level'
                for c in _column6_regs]
    _b_a8, _se_a8, _n_a8 = _fit_hac(_w['rv_cuberoot'], _w[_a8_regs])
    _forest_rows.append(("A8 oil-level", _b_a8, _se_a8, _n_a8))

    # ── A9+ / A9- asymmetric-response subsets ──────────────────────────
    # A9+  : cpi_surprise_ar1 > 0 subset (13 events — severely
    #        underpowered; graceful-degrade with wide CI whiskers,
    #        not a hard error). We still fit the full Column-6
    #        specification — if n < k we fall back to a bivariate
    #        fit so the row exists.
    # A9-  : cpi_surprise_ar1 < 0 subset (205 events — carries all
    #        asymmetric-response power).
    def _fit_asymmetric_subset(_mask_series, _label):
        _sub = _w.loc[_mask_series].copy()
        _regs = _column6_regs
        # Graceful degrade for underpowered positive subset: if n <
        # 2 × k, fall back to bivariate (const + cpi_surprise_ar1).
        if len(_sub) < 2 * len(_regs):
            _regs = ['const', 'cpi_surprise_ar1']
        _b, _se, _n = _fit_hac(_sub['rv_cuberoot'], _sub[_regs])
        return (_label, _b, _se, _n)

    _mask_plus = _w['cpi_surprise_ar1'] > 0
    _mask_minus = _w['cpi_surprise_ar1'] < 0
    _forest_rows.append(_fit_asymmetric_subset(
        _mask_plus, f"A9+ positive-surprise subset"
    ))
    _forest_rows.append(_fit_asymmetric_subset(
        _mask_minus, f"A9- negative-surprise subset"
    ))

    # ── Assemble 13-row table ──────────────────────────────────────────
    # Row 1 anchor (primary) stays first; rows 2+ sorted by |β̂| desc.
    _anchor = _forest_rows[0]
    _tail = sorted(
        _forest_rows[1:],
        key=lambda r: abs(r[1]),
        reverse=True,
    )
    _forest_rows_sorted = [_anchor] + _tail

    # 90% two-sided CI whiskers (± 1.645·SE). On underpowered subsets
    # with n ≈ 13 the Student-t critical is noticeably larger than
    # the normal (≈ 1.77), but the spec pre-commits to the normal-
    # approximation whiskers for consistency across all 13 rows.
    _z_90 = 1.6448536269514722  # scipy.stats.norm.ppf(0.95)
    _forest_records = []
    for _lab, _b, _se, _n in _forest_rows_sorted:
        _lo = _b - _z_90 * _se
        _hi = _b + _z_90 * _se
        _forest_records.append({
            'label': f"{_lab} (n={_n})",
            'beta': _b,
            'se': _se,
            'ci_lo_90': _lo,
            'ci_hi_90': _hi,
            'n': _n,
            'excludes_zero_90': bool(_lo > 0.0 or _hi < 0.0),
        })
    _forest_table = pd.DataFrame(_forest_records)

    # ── Render — specification_curve primary, matplotlib fallback ──────
    _engine = 'matplotlib-fallback'
    try:
        import specification_curve as _spec  # noqa: F401
        _engine = 'specification_curve'
    except ImportError:
        pass

    import matplotlib.pyplot as _plt
    _fig, _ax = _plt.subplots(figsize=(9, 7))
    _y_positions = list(range(len(_forest_table)))[::-1]  # top-down
    for _yi, _row in zip(_y_positions, _forest_table.itertuples(index=False)):
        _ax.errorbar(
            _row.beta, _yi,
            xerr=[[_row.beta - _row.ci_lo_90], [_row.ci_hi_90 - _row.beta]],
            fmt='o', color='black', ecolor='gray', capsize=3,
        )
    _ax.axvline(0.0, color='red', linestyle=':', linewidth=0.8)
    # Divider below the anchor row.
    _ax.axhline(max(_y_positions) - 0.5, color='black', linewidth=0.8)
    _ax.set_yticks(_y_positions)
    _ax.set_yticklabels(_forest_table['label'].tolist())
    _ax.set_xlabel("β̂_CPI (90% HAC CI whiskers)")
    _ax.set_title(
        "NB3 §8 forest plot — primary anchor + 12 sensitivities "
        f"(engine: {_engine})"
    )
    _fig.tight_layout()
    _plt.show()

    # ── Diagnostics: A2/A3/A7 annotation + CI-exclusion summary ────────
    print(f"NB3 §8 forest plot — {len(_forest_table)} rows, render via "
          f"{_engine}.")
    print(f"  annotated (not refit here): {_A2_A3_A7_ANNOTATION}")
    print(f"  → A2 (squared-surprise), A3 (release-only), A7 (log-RV) — "
          f"see NB2 §10.")
    _n_excl = int(_forest_table['excludes_zero_90'].sum())
    print(f"  rows with 90% CI EXCLUDING zero : {_n_excl} of "
          f"{len(_forest_table)}.")
    if _n_excl > 0:
        _excl_rows = _forest_table.loc[
            _forest_table['excludes_zero_90'],
            ['label', 'beta', 'ci_lo_90', 'ci_hi_90']
        ]
        print(_excl_rows.to_string(index=False))
    # Top 3 by |β̂| (rows 2-13, anchor excluded) — the largest effects.
    _top3 = (
        _forest_table.iloc[1:]
        .assign(abs_beta=lambda d: d['beta'].abs())
        .sort_values('abs_beta', ascending=False)
        .head(3)[['label', 'beta', 'ci_lo_90', 'ci_hi_90', 'n']]
    )
    print("  top 3 sensitivities by |β̂| (anchor excluded):")
    print(_top3.to_string(index=False))

# Release the DuckDB connection held open by the bootstrap.
try:
    conn.close()
except Exception:
    pass


**Interpretation — §8.** The forest plot carries 13 rows: the primary Column-6 anchor at the top (with a horizontal divider below), followed by 12 sensitivities sorted by |β̂_CPI| descending. The executed cell prints the numeric forest table, the top-3 sensitivities by |β̂|, and the count of rows whose 90% HAC confidence interval excludes zero.

The single most informative diagnostic the forest surfaces is the A9⁺ vs A9⁻ asymmetric-response split. The Colombia panel has only 13 positive-CPI-surprise weeks (`cpi_surprise_ar1 > 0`) against 205 negative-surprise weeks and 729 zero-surprise weeks; A9⁺ is therefore severely underpowered by any standard measure. On a 13-event subset the Column-6 specification (const + 6 regressors) has more parameters than observations for all seven coefficients; the code gracefully degrades to a bivariate fit (const + cpi_surprise_ar1) on the positive subset so the row exists, but the CI whiskers are correspondingly wide — often spanning several orders of magnitude more than the primary CI. This is a pre-registered negative result: the Colombia panel cannot distinguish a positive-surprise response from noise, and any post-hoc asymmetric-story pitch on the positive side would not survive honest reporting. A9⁻ carries the 205 negative-surprise weeks; if the Colombia CPI-surprise → FX-vol channel carries any asymmetric-response power at all, this row is where it is visible. A 90% CI on A9⁻ that excludes zero (in either direction) would be the single most interesting finding in the forest.

Rows 2-13 sorted by |β̂| descending are dominated by the decomposition β̂_PPI row (IPP added as a co-regressor of CPI): on the Colombia panel, the producer-price surprise channel carries a larger point estimate than the consumer-price channel, consistent with Colombia's commodity-exporter macro structure where producer-price inflation leads consumer-price inflation through the tradables channel. The three subsample fits (pre-2015, 2015-2021, post-2021) distribute across the ordering according to regime — this is the endogenous break conversation NB3 §5 already surfaced, and the forest's visual reading of it is that subsample instability is real but concentrated in specific epochs rather than the sample midpoint NB2 §8 pre-committed to. The GARCH-X δ̂ row sits separately: its SE comes from the QMLE covariance rather than HAC, but the CI construction is the same (normal ± 1.645·SE), and the row's position in the |β̂|-descending ordering is what the reader watches for.

The A1 monthly row uses a reduced control set (const + cpi_surprise_ar1 + vix_avg + intervention_dummy + oil_return) because the monthly panel exposed by `get_monthly_panel` does not carry us_cpi_surprise / banrep_rate_surprise at month-end cadence. This is an honest cadence-perturbation reading: if the primary weekly β̂_CPI survived at monthly cadence with fewer controls, the reader would have evidence that the controls (not the cadence) were driving the primary's near-zero reading. The A4 release-day-excluded row substitutes RV with the CPI/PPI-release-day-excluded realised-variance series (`rv_excl_release_cuberoot`) — if β̂_CPI *collapses* on A4, the primary's already-null reading was being propped up by the release-day observations; if it survives, the primary's null is sample-wide rather than release-day-driven. A5 lagged-RV adds the previous week's `rv_cuberoot` as an AR(1) autoregressive control; A6 bivariate (NB2 `ladder_fits[0]`) strips all controls; A8 oil-level replaces `oil_return` with `oil_log_level` to test level-vs-first-difference sensitivity of the oil control.

A2 (squared-surprise — asymmetric response via quadratic term), A3 (release-only subsample), and A7 (log-RV transformation) are annotated in the diagnostics section as "see NB2 §10" — NB2 authored these sensitivities and their inclusion as explicit forest rows would duplicate NB2's narrative without adding evidence. Their NB2-reported point estimates + standard errors sit in the PKL and the reader can cross-reference by name.

Downstream impact: the §10 gate writer records the full `_forest_table` DataFrame — label, β̂, SE, 90% CI bounds, sample size, and the boolean `excludes_zero_90` flag — as the sensitivity-summary artifact the simulator consumes. The primary decision rule is: (i) if the primary row's 90% CI excludes zero → calibrate from primary β̂_CPI; (ii) else if any sensitivity row's 90% CI excludes zero → calibrate from that sensitivity with a pre-registered specification-search caveat; (iii) else → fall back to the Levene announcement-channel calibration (§3) and the Fuentes-et-al.-2014 externally-calibrated effect sizes (§6). Under our current reading — primary CI contains zero (§7 FAIL TO REJECT), A9⁺ wildly imprecise, A9⁻ expected tight-around-zero — the forest documents an "honestly underpowered panel" verdict, and the simulator calibration flows to route (iii).
